## Notebook for using the CDS API to get ERA5 data

#### Run the pip install to get the latest cdsapi

In [2]:
#!pip install cdsapi
# created .cdsapirc file in ~/ with UID and API-key per https://pypi.org/project/cdsapi/

  Obtaining dependency information for cdsapi from https://files.pythonhosted.org/packages/57/c0/22f3e6f329dd690cc403e21ec894231ffc668ca7925b62de99612530a0a4/cdsapi-0.7.0-py2.py3-none-any.whl.metadata
  Obtaining dependency information for cads-api-client>=0.9.2 from https://files.pythonhosted.org/packages/3e/fd/cffb3fdc5bb3ef55d10973bbc798d8bdea6f3049ab7cdd9895c1183022f4/cads_api_client-1.0.3-py3-none-any.whl.metadata
  Preparing metadata (setup.py) ... done
  Created wheel for multiurl: filename=multiurl-0.3.1-py3-none-any.whl size=21132 sha256=ad4295cb649a642d0117d24c835cd2270b6713216145cfe01c024a8f2f39fe45
  Stored in directory: /Users/tedscott/Library/Caches/pip/wheels/88/54/66/4e1d7b60e01b6f962672d16bc2227061edd0af136327d6f9e4
Successfully built multiurl


In [5]:
import cdsapi
import requests

### This block is for grabbing a single year, entire globe <br><br> This actually errors out and says the request is too large for netcdf, apparently

In [6]:
# c = cdsapi.Client()

# c.retrieve(
#     'reanalysis-era5-single-levels',
#     {
#         'product_type': 'reanalysis',
#         'variable': [
#             '2m_temperature', 'land_sea_mask',
#         ],
#         'year': '1980',
#         'month': [
#             '01', '02', '03',
#             '04', '05', '06',
#             '07', '08', '09',
#             '10', '11', '12',
#         ],
#         'day': [
#             '01', '02', '03',
#             '04', '05', '06',
#             '07', '08', '09',
#             '10', '11', '12',
#             '13', '14', '15',
#             '16', '17', '18',
#             '19', '20', '21',
#             '22', '23', '24',
#             '25', '26', '27',
#             '28', '29', '30',
#             '31',
#         ],
#         'time': [
#             '00:00', '01:00', '02:00',
#             '03:00', '04:00', '05:00',
#             '06:00', '07:00', '08:00',
#             '09:00', '10:00', '11:00',
#             '12:00', '13:00', '14:00',
#             '15:00', '16:00', '17:00',
#             '18:00', '19:00', '20:00',
#             '21:00', '22:00', '23:00',
#         ],
#         'format': 'netcdf',
#     },
#     'ERA5-1980-Full-T2m_LSM-download.nc')

### This block is for specific years and months, specific area lat/lon

In [ ]:
# try multiple years, same region, JJA
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            '2m_temperature', 'land_sea_mask',
        ],
        'year': ['2020','2021','2022','2023'],
        'month': [
            '06', '07', '08',
        ],
        'day': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
            '13', '14', '15',
            '16', '17', '18',
            '19', '20', '21',
            '22', '23', '24',
            '25', '26', '27',
            '28', '29', '30',
            '31',
        ],
        'time': [
            '00:00', '01:00', '02:00',
            '03:00', '04:00', '05:00',
            '06:00', '07:00', '08:00',
            '09:00', '10:00', '11:00',
            '12:00', '13:00', '14:00',
            '15:00', '16:00', '17:00',
            '18:00', '19:00', '20:00',
            '21:00', '22:00', '23:00',
        ],
        'area': [
            50, -124, 47,
            -122,
        ],
        'format': 'netcdf',
    },
    '2020-2023_JJA_PugetSound_T2m-LSM.nc')

### <br> Using the Daily Statistics Application to get daily means instead of pulling hourly data and calculating it <br> https://cds.climate.copernicus.eu/cdsapp#!/software/app-c3s-daily-era5-statistics?tab=overview <br><br> From this forum post: https://forum.ecmwf.int/t/retrieve-daily-era5-era5-land-data-using-the-cds-api/1150/1 <br><br> After running - note each month is roughly 128MB or about 1.5GB per year! <br> And, it took about 10 min total to retrieve 12 months (12 files)

In [13]:
import cdsapi
import requests
 
# CDS API script to use CDS service to retrieve daily ERA5* variables and iterate over
# all months in the specified years.
 
# Requires:
# 1) the CDS API to be installed and working on your system
# 2) You have agreed to the ERA5 Licence (via the CDS web page)
# 3) Selection of required variable, daily statistic, etc
 
# Output:
# 1) separate netCDF file for chosen daily statistic/variable for each month
 
c = cdsapi.Client()#timeout=300)
 
# Uncomment years as required
 
years =  [
#            '1979',
#            '1980', '1981',
#            '1982', '1983', '1984',
#            '1985', '1986', '1987',
#            '1988', '1989', '1990',
#            '1991', '1992', '1993',
#            '1994', '1995', '1996',
#            '1997', '1998', '1999',
#            '2000', '2001', '2002',
#            '2003', '2004', '2005',
#            '2006', '2007', '2008',
#            '2009', '2010', '2011',
#            '2012', '2013', '2014',
#            '2015', '2016', '2017',
#            '2018', '2019', '2020',
            '2021'#,
#             '2022'
]
 
 
# Retrieve all months for a given year.
 
months = ['01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12']
 
# For valid keywords, see Table 2 of:
# https://datastore.copernicus-climate.eu/documents/app-c3s-daily-era5-statistics/C3S_Application-Documentation_ERA5-daily-statistics-v2.pdf
 
# select your variable; name must be a valid ERA5 CDS API name.
var = "2m_temperature"
 
# Select the required statistic, valid names given in link above
stat = "daily_mean"
 
# Loop over years and months
 
for yr in years:
    for mn in months:
        result = c.service(
        "tool.toolbox.orchestrator.workflow",
        params={
             "realm": "user-apps",
             "project": "app-c3s-daily-era5-statistics",
             "version": "master",
             "kwargs": {
                 "dataset": "reanalysis-era5-single-levels",
                 "product_type": "reanalysis",
                 "variable": var,
                 "statistic": stat,
                 "year": yr,
                 "month": mn,
                 "time_zone": "UTC+00:0",
                 "frequency": "1-hourly",
#
# Users can change the output grid resolution and selected area
#
#                "grid": "1.0/1.0",
#                "area":{"lat": [10, 60], "lon": [65, 140]}
 
                 },
        "workflow_name": "application"
        })
         
# set name of output file for each month (statistic, variable, year, month     
 
        file_name = "download_" + stat + "_" + var + "_" + yr + "_" + mn + ".nc"
         
        location=result[0]['location']
        res = requests.get(location, stream = True)
        print("Writing data to " + file_name)
        with open(file_name,'wb') as fh:
            for r in res.iter_content(chunk_size = 1024):
                fh.write(r)
        fh.close()

2024-06-04 12:15:36,317 INFO Welcome to the CDS
2024-06-04 12:15:36,319 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-c17c56e5d8264523a3b68eb44dca2a5d
2024-06-04 12:15:36,703 INFO Request is queued
2024-06-04 12:15:37,879 INFO Request is running
2024-06-04 12:15:50,771 INFO Request is completed


Writing data to download_daily_mean_2m_temperature_2021_01.nc


2024-06-04 12:16:01,055 INFO Welcome to the CDS
2024-06-04 12:16:01,057 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-7cccc096b14541b192bfcb958c835a65
2024-06-04 12:16:01,502 INFO Request is queued
2024-06-04 12:16:02,678 INFO Request is running
2024-06-04 12:16:23,353 INFO Request is completed


Writing data to download_daily_mean_2m_temperature_2021_02.nc


2024-06-04 12:17:18,855 INFO Welcome to the CDS
2024-06-04 12:17:18,857 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-c5475966493b4c69922cddeff482449f
2024-06-04 12:17:19,078 INFO Request is queued
2024-06-04 12:17:20,261 INFO Request is running
2024-06-04 12:17:40,936 INFO Request is completed


...

Writing data to download_daily_mean_2m_temperature_2021_10.nc


2024-06-04 12:26:23,722 INFO Welcome to the CDS
2024-06-04 12:26:23,723 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-3b7fef061e65416d90687984fb481cab
2024-06-04 12:26:23,981 INFO Request is queued
2024-06-04 12:26:25,165 INFO Request is running
2024-06-04 12:26:45,824 INFO Request is completed


Writing data to download_daily_mean_2m_temperature_2021_11.nc


2024-06-04 12:27:02,139 INFO Welcome to the CDS
2024-06-04 12:27:02,141 INFO Sending request to https://cds.climate.copernicus.eu/api/v2/tasks/services/tool/toolbox/orchestrator/workflow/clientid-5f6e03a5119c4282be3fe8fcdbd7994f
2024-06-04 12:27:02,336 INFO Request is queued
2024-06-04 12:27:03,516 INFO Request is running
2024-06-04 12:27:24,184 INFO Request is completed


Writing data to download_daily_mean_2m_temperature_2021_12.nc
